# Execution and source notes

Executed work from the IBM Applied Data Science Capstone labs. Unexecuted starter code cells were removed for a clear run order. Original instructions and authorship remain. The final corrected_map output supersedes earlier proximity maps. Selected shoreline distance: 0.838 km. Selected road distance: 0.644 km. Points are manually selected, not proven nearest points. Railway/city measurements were not performed.

<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDS0321ENSkillsNetwork26802033-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Hands-on Lab: Interactive Visual Analytics with Folium**


Estimated time needed: **40** minutes


The launch success rate may depend on many factors such as payload mass, orbit type, and so on. It may also depend on the location and proximities of a launch site, i.e., the initial position of rocket trajectories. Finding an optimal location for building a launch site certainly involves many factors and hopefully we could discover some of the factors by analyzing the existing launch site locations.


In the previous exploratory data analysis labs, you have visualized the SpaceX launch dataset using `matplotlib` and `seaborn` and discovered some preliminary correlations between the launch site and success rates. In this lab, you will be performing more interactive visual analytics using `Folium`.


## Objectives


This lab contains the following tasks:

*   **TASK 1:** Mark all launch sites on a map
*   **TASK 2:** Mark the success/failed launches for each site on the map
*   **TASK 3:** Calculate the distances between a launch site to its proximities

After completed the above tasks, you should be able to find some geographical patterns about launch sites.


Let's first import required Python packages for this lab:


In [3]:
import piplite

print("Installing Folium...", flush=True)
await piplite.install("folium")
print("Folium installed.", flush=True)

print("Installing pandas...", flush=True)
await piplite.install("pandas")
print("Setup complete.", flush=True)

Installing Folium...
Folium installed.
Installing pandas...
Setup complete.


In [4]:
import folium
import pandas as pd
import io

from js import fetch
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon

url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
)

response = await fetch(url)
csv_text = await response.text()
spacex_df = pd.read_csv(io.StringIO(csv_text))

spacex_df = spacex_df[["Launch Site", "Lat", "Long", "class"]]

launch_sites_df = (
    spacex_df.groupby("Launch Site", as_index=False)
    .first()[["Launch Site", "Lat", "Long"]]
)

print("Loaded launches:", len(spacex_df))
print("Launch sites:", len(launch_sites_df))

Loaded launches: 56
Launch sites: 4


If you need to refresh your memory about folium, you may download and refer to this previous folium lab:


[Generating Maps with Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/v4/DV0101EN-Exercise-Generating-Maps-in-Python.ipynb)


First, let's try to add each site's location on a map using site's latitude and longitude coordinates


The following dataset with the name `spacex_launch_geo.csv` is an augmented dataset with latitude and longitude added for each site.


Now, you can take a look at what are the coordinates for each site.


Above coordinates are just plain numbers that can not give you any intuitive insights about where are those launch sites. If you are very good at geography, you can interpret those numbers directly in your mind. If not, that's fine too. Let's visualize those locations by pinning them on a map.


We first need to create a folium `Map` object, with an initial center location to be NASA Johnson Space Center at Houston, Texas.


We could use `folium.Circle` to add a highlighted circle area with a text label on a specific coordinate. For example,


and you should find a small yellow circle near the city of Houston and you can zoom-in to see a larger circle.


Now, let's add a circle for each launch site in data frame `launch_sites`


*TODO:*  Create and add `folium.Circle` and `folium.Marker` for each launch site on the site map


An example of folium.Circle:


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


An example of folium.Marker:


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [6]:
site_map = folium.Map(
    location=[30, -95],
    zoom_start=4
)

for _, row in launch_sites_df.iterrows():
    coordinate = [row["Lat"], row["Long"]]
    name = row["Launch Site"]

    folium.Circle(
        location=coordinate,
        radius=1000,
        color="blue",
        fill=True,
        popup=name
    ).add_to(site_map)

    folium.Marker(
        location=coordinate,
        tooltip=name,
        icon=DivIcon(
            icon_size=(160, 30),
            icon_anchor=(0, 0),
            html=(
                '<div style="font-size:12px; color:blue; '
                'white-space:nowrap;">'
                f"<b>{name}</b></div>"
            )
        )
    ).add_to(site_map)

site_map.fit_bounds(
    launch_sites_df[["Lat", "Long"]].values.tolist()
)

site_map



The generated map with marked launch sites should look similar to the following:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Now, you can explore the map by zoom-in/out the marked areas
, and try to answer the following questions:

*   Are all launch sites in proximity to the Equator line?
*   Are all launch sites in very close proximity to the coast?

Also please try to explain your findings.


Next, let's try to enhance the map by adding the launch outcomes for each site, and see which sites have high success rates.
Recall that data frame spacex_df has detailed launch records, and the `class` column indicates if this launch was successful or not


Next, let's create markers for all launch records.
If a launch was successful `(class=1)`, then we use a green marker and if a launch was failed, we use a red marker `(class=0)`


Note that a launch only happens in one of the four launch sites, which means many launch records will have the exact same coordinate. Marker clusters can be a good way to simplify a map containing many markers having the same coordinate.


Let's first create a `MarkerCluster` object


*TODO:* Create a new column in `spacex_df` dataframe called `marker_color` to store the marker colors based on the `class` value


*TODO:* For each launch result in `spacex_df` data frame, add a `folium.Marker` to `marker_cluster`


In [7]:
spacex_df["marker_color"] = spacex_df["class"].map({
    1: "green",
    0: "red"
})

marker_cluster = MarkerCluster().add_to(site_map)

for index, row in spacex_df.iterrows():
    outcome = "Success" if row["class"] == 1 else "Unsuccessful"

    folium.Marker(
        location=[row["Lat"], row["Long"]],
        popup=f"{row['Launch Site']}: {outcome}",
        tooltip=f"Launch record {index}: {outcome}",
        icon=folium.Icon(
            color=row["marker_color"],
            icon="info-sign"
        )
    ).add_to(marker_cluster)

site_map

Your updated map may look like the following screenshots:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


From the color-labeled markers in marker clusters, you should be able to easily identify which launch sites have relatively high success rates.


Next, we need to explore and analyze the proximities of launch sites.


Let's first add a `MousePosition` on the map to get coordinate for a mouse over a point on the map. As such, while you are exploring the map, you can easily find the coordinates of any points of interests (such as railway)


In [8]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"

MousePosition(
    position="topright",
    separator=" Long: ",
    prefix="Lat:",
    lat_formatter=formatter,
    lng_formatter=formatter
).add_to(site_map)

site_map

Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


Now zoom in to a launch site and explore its proximity to see if you can easily find any railway, highway, coastline, etc. Move your mouse to these points and mark down their coordinates (shown on the top-left) in order to the distance to the launch site.


In [9]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # approximate radius of earth in km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

*TODO:* Mark down a point on the closest coastline using MousePosition and calculate the distance between the coastline point and the launch site.


In [10]:
selected_site = launch_sites_df[
    launch_sites_df["Launch Site"] == "CCAFS SLC-40"
].iloc[0]

launch_coordinate = [
    float(selected_site["Lat"]),
    float(selected_site["Long"])
]

coastline_coordinate = [28.56367, -80.57163]

distance_coastline = calculate_distance(
    launch_coordinate[0],
    launch_coordinate[1],
    coastline_coordinate[0],
    coastline_coordinate[1]
)

print("Launch site:", selected_site["Launch Site"])
print(
    "Distance to example coastline point:",
    round(distance_coastline, 3),
    "km"
)

Launch site: CCAFS SLC-40
Distance to example coastline point: 0.51 km


In [11]:
coast_map = folium.Map(
    location=launch_coordinate,
    zoom_start=15
)

folium.Marker(
    launch_coordinate,
    tooltip="CCAFS SLC-40",
    popup="Launch site"
).add_to(coast_map)

folium.Marker(
    coastline_coordinate,
    tooltip="Selected coastline point",
    icon=folium.Icon(color="orange")
).add_to(coast_map)

folium.Marker(
    coastline_coordinate,
    icon=DivIcon(
        icon_size=(180, 30),
        html=(
            '<div style="font-size:14px; color:darkred; '
            'white-space:nowrap;">'
            f"<b>{distance_coastline:.2f} km to launch site</b>"
            "</div>"
        )
    )
).add_to(coast_map)

coast_map

*TODO:* Draw a `PolyLine` between a launch site to the selected coastline point


In [12]:
folium.PolyLine(
    locations=[launch_coordinate, coastline_coordinate],
    color="red",
    weight=3,
    tooltip=f"Straight-line distance: {distance_coastline:.2f} km"
).add_to(coast_map)

coast_map

Your updated map with distance line should look like the following screenshot:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


*TODO:* Similarly, you can draw a line betwee a launch site to its closest city, railway, highway, etc. You need to use `MousePosition` to find the their coordinates on the map first


A railway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


A highway map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


A city map symbol may look like this:


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [14]:
MousePosition(
    position="bottomleft",
    separator=" / ",
    prefix="Latitude / Longitude:",
    num_digits=5
).add_to(coast_map)

coast_map


In [18]:
road_coordinate = [28.56152, -80.57051]

distance_road = calculate_distance(
    launch_coordinate[0],
    launch_coordinate[1],
    road_coordinate[0],
    road_coordinate[1]
)

folium.Marker(
    road_coordinate,
    tooltip="Selected point on Samuel C. Phillips Parkway",
    popup=f"Distance from launch site: {distance_road:.3f} km",
    icon=folium.Icon(color="purple")
).add_to(coast_map)

folium.PolyLine(
    locations=[launch_coordinate, road_coordinate],
    color="purple",
    weight=3,
    tooltip=f"Distance to selected road point: {distance_road:.3f} km"
).add_to(coast_map)

print("Distance to selected road point:", round(distance_road, 3), "km")
display(coast_map)

Distance to selected road point: 0.644 km


In [19]:
coastline_coordinate = [28.56415, -80.56831]

distance_coastline = calculate_distance(
    launch_coordinate[0],
    launch_coordinate[1],
    coastline_coordinate[0],
    coastline_coordinate[1]
)

corrected_map = folium.Map(
    location=launch_coordinate,
    zoom_start=15
)

folium.Marker(
    launch_coordinate,
    tooltip="CCAFS SLC-40"
).add_to(corrected_map)

points = [
    ("Selected shoreline point", coastline_coordinate,
     distance_coastline, "orange"),
    ("Selected road point", road_coordinate,
     distance_road, "purple")
]

for label, coordinate, distance, color in points:
    folium.Marker(
        coordinate,
        popup=f"{label}: {distance:.3f} km from launch site",
        tooltip=f"{label}: {distance:.3f} km",
        icon=folium.Icon(color=color)
    ).add_to(corrected_map)

    folium.PolyLine(
        locations=[launch_coordinate, coordinate],
        color=color,
        weight=3,
        tooltip=f"{label}: {distance:.3f} km"
    ).add_to(corrected_map)

print("Distance to selected shoreline point:",
      round(distance_coastline, 3), "km")
print("Distance to selected road point:",
      round(distance_road, 3), "km")

display(corrected_map)

Distance to selected shoreline point: 0.838 km
Distance to selected road point: 0.644 km


After you plot distance lines to the proximities, you can answer the following questions easily:

*   Are launch sites in close proximity to railways?
*   Are launch sites in close proximity to highways?
*   Are launch sites in close proximity to coastline?
*   Do launch sites keep certain distance away from cities?

Also please try to explain your findings.


# Next Steps:

Now you have discovered many interesting insights related to the launch sites' location using folium, in a very interactive way. Next, you will need to build a dashboard using Ploty Dash on detailed launch records.


## Authors


[Pratiksha Verma](https://www.linkedin.com/in/pratiksha-verma-6487561b1/)


<!--## Change Log--!>


<!--| Date (YYYY-MM-DD) | Version | Changed By      | Change Description      |
| ----------------- | ------- | -------------   | ----------------------- |
| 2022-11-09        | 1.0     | Pratiksha Verma | Converted initial version to Jupyterlite|--!>


### <h3 align="center"> IBM Corporation 2022. All rights reserved. <h3/>
